# 🩺 Protocolos Clínicos e Diretrizes Terapêuticas - PCDT

O script abaixo extrai todos os Protocolos Clínicos e Diretrizes Terapêuticas (PCDT) contidos no portal do Ministério da Saúde (https://www.gov.br/saude/pt-br/assuntos/pcdt).

Os PCDTs são diretrizes clínicas do Sistema Único de Saúde (SUS) que norteiam condutas no cuidado às doenças e agravos, auxiliando médicos, profissionais de saúde, pacientes, estabelecimentos e serviços de saúde referenciais e os gestores em saúde.

## 📥 Crawler e Downloader

In [1]:
import os
import time
import requests
from bs4 import BeautifulSoup
import string
from urllib.parse import urljoin, unquote
import urllib3
import re

# Desabilita avisos de certificado SSL (comum no Gov.br)
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

BASE_URL = "https://www.gov.br/saude/pt-br/assuntos/pcdt/"
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
}
MAIN_DIR = "PCDTs_Ministerio_da_Saude"

def limpar_nome_arquivo(nome):
    """Remove caracteres que não são permitidos em nomes de arquivos pelo Windows/Linux."""
    return re.sub(r'[\\/*?:"<>|]', "", nome)

def extrair_nome_arquivo(response, nome_padrao):
    """Extrai e decodifica o nome do arquivo enviado pelo cabeçalho do Plone."""
    cd = response.headers.get("Content-Disposition", "")
    if not cd:
        return nome_padrao + ".pdf"

    # Tenta capturar no padrão codificado do Plone: filename*=UTF-8''nome%20com%20espaco.pdf
    match_utf8 = re.search(r"filename\*=UTF-8''(.+)", cd, re.IGNORECASE)
    if match_utf8:
        nome_decodificado = unquote(match_utf8.group(1)) # Transforma %20 em espaço, etc.
        return limpar_nome_arquivo(nome_decodificado)

    # Tenta capturar no padrão normal: filename="nome_do_arquivo.pdf"
    match_normal = re.search(r'filename=["\']?([^"\';]+)["\']?', cd, re.IGNORECASE)
    if match_normal:
        return limpar_nome_arquivo(match_normal.group(1).strip())

    return nome_padrao + ".pdf"

def main():
    if not os.path.exists(MAIN_DIR):
        os.makedirs(MAIN_DIR)

    letras = string.ascii_lowercase

    for letra in letras:
        url_letra = urljoin(BASE_URL, f"{letra}/")
        url_letra_limpa = url_letra.rstrip("/")

        letra_dir = os.path.join(MAIN_DIR, letra)
        if not os.path.exists(letra_dir):
            os.makedirs(letra_dir)

        print(f"\n[{letra.upper()}] Acessando página: {url_letra}")

        try:
            response = requests.get(url_letra, headers=HEADERS, timeout=15, verify=False)
            response.raise_for_status()
        except requests.RequestException as e:
            print(f"Erro ao acessar a página '{letra}': {e}")
            continue

        soup = BeautifulSoup(response.content, "html.parser")
        links_documentos = set()

        for a_tag in soup.find_all("a", href=True):
            href = a_tag["href"]

            # Limpa parâmetros de busca e âncoras da URL
            clean_href = href.split("?")[0].split("#")[0].rstrip("/")

            # Se o Plone jogou um /view no final do link, nós o removemos
            if clean_href.endswith("/view"):
                clean_href = clean_href[:-5]

            # Se o link já veio com /@@download/file, nós o removemos temporariamente para padronizar
            if clean_href.endswith("/@@download/file"):
                clean_href = clean_href.replace("/@@download/file", "")

            # Garante que o link é um "filho" da letra (ex: ../pcdt/a/doenca) e não a própria letra (../pcdt/a)
            if clean_href.startswith(url_letra_limpa + "/") and len(clean_href) > len(url_letra_limpa) + 1:
                links_documentos.add(clean_href)

        if not links_documentos:
            print(f"Nenhum documento encontrado na letra '{letra}'.")
            continue

        # Inicia o download
        for base_link in sorted(links_documentos):
            download_url = f"{base_link}/@@download/file"
            slug_doenca = base_link.split('/')[-1]

            print(f"  - Obtendo: {slug_doenca} ...", end=" ", flush=True)

            try:
                pdf_response = requests.get(download_url, headers=HEADERS, timeout=30, stream=True, verify=False)

                if pdf_response.status_code == 200:
                    content_type = pdf_response.headers.get('Content-Type', '').lower()

                    if 'text/html' not in content_type:
                        nome_arquivo = extrair_nome_arquivo(pdf_response, slug_doenca)

                        if not nome_arquivo.lower().endswith('.pdf'):
                            nome_arquivo += '.pdf'

                        caminho_arquivo = os.path.join(letra_dir, nome_arquivo)

                        if os.path.exists(caminho_arquivo):
                            print(f"Já existe! (Pulando)")
                            continue

                        with open(caminho_arquivo, "wb") as f:
                            for chunk in pdf_response.iter_content(chunk_size=8192):
                                if chunk:
                                    f.write(chunk)
                        print(f"OK! (Salvo: {nome_arquivo})")
                    else:
                        print("FALHA! (Retornou HTML - Possível subpasta sem arquivo direto)")
                else:
                    print(f"FALHA! (HTTP {pdf_response.status_code})")

            except requests.RequestException as e:
                print(f"ERRO! ({e})")

            time.sleep(0.5)

if __name__ == "__main__":
    main()


[A] Acessando página: https://www.gov.br/saude/pt-br/assuntos/pcdt/a/
  - Obtendo: a ... FALHA! (HTTP 404)
  - Obtendo: acidente-vascular-cerebral-isquemico-agudo ... OK! (Salvo: Acidente Vascular Cerebral Isquêmico Agudo - Portaria Conjunta SAES-SECTICS n 29.pdf)
  - Obtendo: acidentes-escorpionicos ... OK! (Salvo: PCDT - Acidentes Escorpiônicos.pdf)
  - Obtendo: acidentes-ofidicos ... FALHA! (HTTP 404)
  - Obtendo: acromegalia.pdf ... OK! (Salvo: Acromegalia - PCDT.pdf)
  - Obtendo: adenocarcinoma-de-colon-e-de-reto ... OK! (Salvo: Adenocarcinoma de Cólon e de Reto - PCDT.pdf)
  - Obtendo: adenocarcinoma-de-estomago.pdf ... OK! (Salvo: Adenocarcinoma de Estomago.pdf)
  - Obtendo: adenocarcinoma-prostata.pdf ... OK! (Salvo: pcdt-Adenocarcinoma-Prostata.pdf)
  - Obtendo: amiloidoses-associadas-a-transtirretina.pdf ... OK! (Salvo: Amiloidoses Associadas a Transtirretina.pdf)
  - Obtendo: anemia-deficiencia-de-ferro ... OK! (Salvo: Anemia Deficiencia de Ferro.pdf)
  - Obtendo: anemia-he

## 📦 Compactação e Download

In [2]:
import shutil
from google.colab import files
import os

MAIN_DIR = "PCDTs_Ministerio_da_Saude"

if os.path.exists(MAIN_DIR):
    print("Compactando os arquivos... Isso pode levar alguns instantes.")
    # Compacta a pasta inteira
    shutil.make_archive('PCDTs_Completos', 'zip', MAIN_DIR)

    print("Iniciando o download do arquivo ZIP...")
    # Solicita ao Colab que empurre o arquivo para o seu PC
    files.download('PCDTs_Completos.zip')
else:
    print("A pasta de PDFs não foi encontrada. Certifique-se de que o Bloco 1 rodou com sucesso!")

Compactando os arquivos... Isso pode levar alguns instantes.
Iniciando o download do arquivo ZIP...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## ⚙️ Instalação das dependências (Célula de Código)



In [3]:
# Instalação das bibliotecas para conversão de PDF para Markdown e fragmentação semântica
!pip install -q pymupdf4llm langchain-text-splitters

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 176.4/176.4 kB 17.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.8/25.8 MB 104.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.9/42.9 MB 64.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 109.4 MB/s eta 0:00:00


## 🔄 Conversão, Enriquecimento e Fragmentação (Célula de Código)

In [4]:
import os
import multiprocessing
from concurrent.futures import ProcessPoolExecutor, as_completed
import pymupdf4llm
from langchain_text_splitters import MarkdownHeaderTextSplitter

MAIN_DIR = "PCDTs_Ministerio_da_Saude"
OUT_DIR = "PCDTs_Markdown_Processados"

if not os.path.exists(OUT_DIR):
    os.makedirs(OUT_DIR)

# Configuração do divisor semântico
headers_to_split_on = [
    ("#", "Header 1"),
    ("##", "Header 2"),
    ("###", "Header 3"),
]
markdown_splitter = MarkdownHeaderTextSplitter(headers_to_split_on=headers_to_split_on)

def processar_unico_pdf(caminho_pdf):
    """Função worker executada em paralelo para cada arquivo PDF."""
    file = os.path.basename(caminho_pdf)
    nome_doenca = file.replace(".pdf", "").replace("- PCDT", "").strip()
    caminho_saida = os.path.join(OUT_DIR, f"{nome_doenca}.md")

    # Pula se o arquivo já foi convertido em execuções anteriores
    if os.path.exists(caminho_saida) and os.path.getsize(caminho_saida) > 0:
        return (nome_doenca, 0, True)

    try:
        # Extrai markdown mantendo estrutura
        md_text = pymupdf4llm.to_markdown(caminho_pdf)

        # Divisão por seções/cabeçalhos
        md_header_splits = markdown_splitter.split_text(md_text)

        # Gravação do arquivo individual
        with open(caminho_saida, "w", encoding="utf-8") as f:
            for chunk in md_header_splits:
                chunk.metadata["doenca"] = nome_doenca
                chunk.metadata["arquivo_origem"] = file
                f.write(f"<!-- METADADOS: doenca: {nome_doenca} | secao: {chunk.metadata.get('Header 1', 'Geral')} -->\n")
                f.write(chunk.page_content)
                f.write("\n\n---\n\n")

        return (nome_doenca, len(md_header_splits), False)
    except Exception as e:
        return (nome_doenca, f"Erro: {str(e)}", False)

# 1. Coleta a lista completa de caminhos dos PDFs
lista_pdfs = []
for root, dirs, files in os.walk(MAIN_DIR):
    for file in files:
        if file.lower().endswith(".pdf"):
            lista_pdfs.append(os.path.join(root, file))

# 2. Identifica quantos núcleos de CPU estão disponíveis
num_cores = multiprocessing.cpu_count()
print(f"🚀 Iniciando processamento paralelo utilizando {num_cores} núcleos de CPU...")
print(f"Total de arquivos encontrados: {len(lista_pdfs)}\n")

arquivos_novos = 0
chunks_totais = 0

# 3. Executa a conversão distribuindo os arquivos entre os núcleos de CPU
with ProcessPoolExecutor(max_workers=num_cores) as executor:
    futures = {executor.submit(processar_unico_pdf, pdf): pdf for pdf in lista_pdfs}

    for future in as_completed(futures):
        nome_doenca, resultado, ja_existia = future.result()
        if ja_existia:
            print(f"⏩ {nome_doenca}: Já processado anteriormente (Pulado).")
        elif isinstance(resultado, int):
            arquivos_novos += 1
            chunks_totais += resultado
            print(f"✅ {nome_doenca}: Processado ({resultado} chunks gerados).")
        else:
            print(f"❌ {nome_doenca}: {resultado}")

print(f"\n Processamento concluído com sucesso!")
print(f"Novos arquivos processados nesta rodada: {arquivos_novos}")
print(f"Os arquivos Markdown estão consolidados em: '{OUT_DIR}'")

🚀 Iniciando processamento paralelo utilizando 8 núcleos de CPU...
Total de arquivos encontrados: 177


=== Document parser messages ===
Using Tesseract for OCR processing.
✅ Palivizumabe para a Prevenção da Infecção pelo Vírus Sincicial Respiratório: Processado (17 chunks gerados).


/usr/local/lib/python3.13/dist-packages/pymupdf4llm/ocr/compute_ocr_features.py:236: RuntimeWarning: Mean of empty slice.
  fft_ratio = magnitude[magnitude > magnitude.mean()].mean() / (
/usr/local/lib/python3.13/dist-packages/numpy/_core/_methods.py:147: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)



=== Document parser messages ===
Using Tesseract for OCR processing.
OCR on page.number=0/1.
OCR on page.number=12/13.
✅ Puberdade Precoce Central: Processado (25 chunks gerados).

=== Document parser messages ===
Using Tesseract for OCR processing.

=== Document parser messages ===
Using Tesseract for OCR processing.
OCR on page.number=0/1.
✅ Pessoas com Deficiência Auditiva: Processado (21 chunks gerados).
✅ Profilaxia primária em caso de Hemofilia Grave: Processado (52 chunks gerados).

=== Document parser messages ===
Using Tesseract for OCR processing.
OCR on page.number=3/4.
OCR on page.number=8/9.
OCR on page.number=22/23.
OCR on page.number=23/24.
✅ Profilaxia Pós-Exposição de Risco (PEP) à Infecção pelo HIV: Processado (56 chunks gerados).

=== Document parser messages ===
Using Tesseract for OCR processing.
OCR on page.number=2/3.
OCR on page.number=7/8.
OCR on page.number=9/10.
OCR on page.number=31/32.
OCR on page.number=32/33.
✅ Porfirias: Processado (42 chunks gerados).


/usr/local/lib/python3.13/dist-packages/pymupdf4llm/ocr/compute_ocr_features.py:236: RuntimeWarning: Mean of empty slice.
  fft_ratio = magnitude[magnitude > magnitude.mean()].mean() / (
/usr/local/lib/python3.13/dist-packages/numpy/_core/_methods.py:147: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)



=== Document parser messages ===
Using Tesseract for OCR processing.
OCR on page.number=21/22.
✅ Psoríase: Processado (74 chunks gerados).

=== Document parser messages ===
Using Tesseract for OCR processing.
OCR on page.number=0/1.
OCR on page.number=2/3.
OCR on page.number=15/16.
OCR on page.number=47/48.
✅ Epidermólise Bolhosa (Diretriz Brasileira): Processado (62 chunks gerados).

=== Document parser messages ===
Using Tesseract for OCR processing.
OCR on page.number=0/1.
✅ PCDT Epilepsia: Processado (61 chunks gerados).

=== Document parser messages ===
Using Tesseract for OCR processing.
OCR on page.number=4/5.
OCR on page.number=22/23.
OCR on page.number=28/29.
OCR on page.number=32/33.
OCR on page.number=33/34.
OCR on page.number=35/36.
OCR on page.number=36/37.
OCR on page.number=37/38.
✅ Portaria Conjunta nº 54 - Endometriose: Processado (64 chunks gerados).

=== Document parser messages ===
Using Tesseract for OCR processing.
OCR on page.number=0/1.
OCR on page.number=36/37

...


=== Document parser messages ===
Using Tesseract for OCR processing.
OCR on page.number=3/4.
OCR on page.number=10/11.
OCR on page.number=25/26.
OCR on page.number=26/27.
OCR on page.number=27/28.
OCR on page.number=28/29.
OCR on page.number=33/34.
✅ Degeneração Macular Relacionada com a Idade - Portaria Conjunta n  24: Processado (57 chunks gerados).

=== Document parser messages ===
Using Tesseract for OCR processing.
OCR on page.number=24/25.
OCR on page.number=38/39.
✅ PCDT da Doença Pulmonar Obstrutiva Crônica: Processado (54 chunks gerados).

=== Document parser messages ===
Using Tesseract for OCR processing.
OCR on page.number=39/40.
OCR on page.number=42/43.
✅ Diagnóstico e tratamento de intoxicações por agrotóxicos - Capítulo 5: Processado (42 chunks gerados).

=== Document parser messages ===
Using Tesseract for OCR processing.
OCR on page.number=6/7.
OCR on page.number=22/23.
OCR on page.number=27/28.
OCR on page.number=34/35.
OCR on page.number=36/37.
✅ Glaucoma: Processa

/usr/local/lib/python3.13/dist-packages/pymupdf4llm/ocr/compute_ocr_features.py:236: RuntimeWarning: Mean of empty slice.
  fft_ratio = magnitude[magnitude > magnitude.mean()].mean() / (
/usr/local/lib/python3.13/dist-packages/numpy/_core/_methods.py:147: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)


✅ Leiomioma de útero: Processado (26 chunks gerados).

=== Document parser messages ===
Using Tesseract for OCR processing.
OCR on page.number=0/1.
✅ Leucemia Mieloide Aguda de Crianças e Adolescentes - Diretrizes Diagnósticas e Terapêuticas: Processado (14 chunks gerados).

=== Document parser messages ===
Using Tesseract for OCR processing.
OCR on page.number=0/1.
OCR on page.number=14/15.
OCR on page.number=16/17.
OCR on page.number=18/19.
OCR on page.number=19/20.
OCR on page.number=30/31.
OCR on page.number=33/34.
OCR on page.number=34/35.
OCR on page.number=67/68.
✅ Linfoma de Hodgkin no Adulto: Processado (54 chunks gerados).

=== Document parser messages ===
Using Tesseract for OCR processing.
OCR on page.number=0/1.
OCR on page.number=20/21.
OCR on page.number=24/25.
OCR on page.number=28/29.
OCR on page.number=36/37.
OCR on page.number=40/41.
OCR on page.number=44/45.
OCR on page.number=47/48.
OCR on page.number=53/54.
OCR on page.number=58/59.
OCR on page.number=61/62.
OCR o

Process ForkProcess-2:
Process ForkProcess-4:
Process ForkProcess-3:
Process ForkProcess-7:
Process ForkProcess-5:
Process ForkProcess-8:
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
Process ForkProcess-1:
Traceback (most recent call last):
Traceback (most recent call last):
  File "/usr/lib/python3.13/multiprocessing/process.py", line 313, in _bootstrap
    self.run()
    ~~~~~~~~^^
  File "/usr/lib/python3.13/multiprocessing/process.py", line 313, in _bootstrap
    self.run()
    ~~~~~~~~^^
  File "/usr/lib/python3.13/multiprocessing/process.py", line 313, in _bootstrap
    self.run()
    ~~~~~~~~^^
  File "/usr/lib/python3.13/multiprocessing/process.py", line 313, in _bootstrap
    self.run()
    ~~~~~~~~^^
  File "/usr/lib/python3.13/multiprocessing/process.py", line 313, in _bootstrap
    self.run()
    ~~~~~~~~^^
  File "/usr/lib/python3.13/multiprocessing/process.py", line 108, in run


KeyboardInterrupt: 

## 💾 Download dos Markdowns processados



In [7]:
import shutil
from google.colab import files

if os.path.exists(OUT_DIR):
    print("Compactando os arquivos Markdown...")
    shutil.make_archive('PCDTs_Markdown', 'zip', OUT_DIR)

    print("Iniciando o download do arquivo ZIP...")
    files.download('PCDTs_Markdown.zip')

Compactando os arquivos Markdown...
Iniciando o download do arquivo ZIP...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>